In [1]:
import pandas as pd
import torch
import torch.nn as nn

In [2]:
df = pd.read_csv('WineQT.csv')
df.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,Id
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,0
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,1
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,2
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6,3
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,4


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1143 entries, 0 to 1142
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         1143 non-null   float64
 1   volatile acidity      1143 non-null   float64
 2   citric acid           1143 non-null   float64
 3   residual sugar        1143 non-null   float64
 4   chlorides             1143 non-null   float64
 5   free sulfur dioxide   1143 non-null   float64
 6   total sulfur dioxide  1143 non-null   float64
 7   density               1143 non-null   float64
 8   pH                    1143 non-null   float64
 9   sulphates             1143 non-null   float64
 10  alcohol               1143 non-null   float64
 11  quality               1143 non-null   int64  
 12  Id                    1143 non-null   int64  
dtypes: float64(11), int64(2)
memory usage: 116.2 KB


In [4]:
df.isnull().sum()

,0
fixed acidity,0
volatile acidity,0
citric acid,0
residual sugar,0
chlorides,0
free sulfur dioxide,0
total sulfur dioxide,0
density,0
pH,0
sulphates,0


In [10]:
df.dtypes
print(df.columns)

Index(['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar',
       'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density',
       'pH', 'sulphates', 'alcohol', 'quality', 'Id'],
      dtype='object')


In [11]:
df = df.drop('Id', axis = 1)

In [13]:
x = torch.tensor(df.drop('quality', axis =1).values, dtype = torch.float32)
y = torch.tensor(df['quality'].values, dtype = torch.float32)

In [14]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x, y,
                                                    test_size = 0.2,
                                                    random_state = 42)

In [21]:
print(x.shape)
print(y.shape)

y = y.reshape(-1, 1)
print(y.shape)

torch.Size([1143, 11])
torch.Size([1143])
torch.Size([1143, 1])


In [23]:
from sklearn.preprocessing import StandardScaler

scaler_x = StandardScaler()
scaler_y = StandardScaler()

x_train_norm = torch.tensor(scaler_x.fit_transform(x_train.numpy()), dtype = torch.float32)
x_test_norm = torch.tensor(scaler_x.transform(x_test.numpy()), dtype = torch.float32)

y_train_norm = torch.tensor(scaler_y.fit_transform(y_train.numpy().reshape(-1, 1)), dtype = torch.float32)
y_test_norm = torch.tensor(scaler_y.transform(y_test.numpy().reshape(-1, 1)), dtype = torch.float32)

In [24]:
print(x_train_norm.shape)

torch.Size([914, 11])


In [39]:
model = nn.Sequential(
    nn.Linear(11, 64),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(64,32),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(32, 1)
)

In [40]:
loss = nn.MSELoss()
opti = torch.optim.Adam(model.parameters(), lr = 0.001)

In [43]:
for epochs in range(5000):
  pred = model(x_train_norm)
  Loss = loss(pred, y_train_norm)
  opti.zero_grad()
  Loss.backward()
  opti.step()

  if epochs % 500 == 0:
    print(f'Epochs {epochs} Loss: {Loss.item():.4f}')

Epochs 0 Loss: 0.3132
Epochs 500 Loss: 0.2954
Epochs 1000 Loss: 0.2749
Epochs 1500 Loss: 0.2547
Epochs 2000 Loss: 0.2639
Epochs 2500 Loss: 0.2668
Epochs 3000 Loss: 0.2741
Epochs 3500 Loss: 0.2375
Epochs 4000 Loss: 0.2338
Epochs 4500 Loss: 0.2598


In [46]:
model.eval()  # ← switches off dropout during testing
with torch.no_grad():
    test_pred = model(x_test_norm)
    test_loss = loss(test_pred, y_test_norm)
    print(f"Test Loss: {test_loss.item():.4f}")

Test Loss: 0.5274


In [47]:
model.eval()
with torch.no_grad():
    sample = x_test_norm[0].unsqueeze(0)
    pred_norm = model(sample)
    pred_quality = scaler_y.inverse_transform(pred_norm.numpy())
    actual_quality = scaler_y.inverse_transform(y_test_norm[0].numpy().reshape(1,-1))
    print(f"Predicted Quality: {pred_quality[0][0]:.2f}")
    print(f"Actual Quality:    {actual_quality[0][0]:.2f}")

Predicted Quality: 5.24
Actual Quality:    5.00
